# LCSE analysis notebook (paper figures)

Analysis behind Figures 3 to 6 and S2, S7 to S13 of *Efficient and Accurate Ligand Strain Calculations in Solution with the AIMNet2 Neural Network Potential*
(Gokcan and Isayev, JCIM 2026): LCSE distributions by net charge, aromatic proportion, rotatable bonds and heavy atoms for all ligands, non-cofactors,
cofactors, single-target and multi-target ligands, and by enzyme class and subclass.

Input: `data/LCSE_details.json.gz` (loaded through `lcse.DATA_DIR`). Run all cells; figures are written to the working directory when `fname` is set in the plotting calls.
Requires `matplotlib` and `seaborn` (`pip install -e .[analysis]`).

In [ ]:
import numpy as np
import sys, os
import json
import glob
import pandas as pd
import statistics

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns 
from matplotlib.patches import Polygon
from matplotlib import rcParams
labelsize = 12
rcParams['xtick.labelsize'] = labelsize
rcParams['ytick.labelsize'] = labelsize
rcParams['font.family'] = 'serif'
plt.rcParams['figure.dpi'] = 300

from matplotlib.colors import ListedColormap, LinearSegmentedColormap

font = {'family': 'serif',
        'color':  'black',
        'weight': 'normal',
        'size': 18,
        }

font2 = {'family': 'serif',
        'color':  'darkviolet',
        'weight': 'normal',
        'size': 18,
        }

distinct_colors=["#F06292","#80DEEA","#00897B","#7E57C2","#F9A825","#0D47A1","#B4BE50","#F4511E","#E91E63","#EF9A9A"]

import warnings
warnings.filterwarnings("ignore")


In [ ]:
class result_loader(object):
   def __init__(self):
 
       self.ligids=None 
       
       self.pdbid=None
       self.ligand_code=None
       self.EC_no=None
       self.enzyme_class=None
       self.enzyme_subclass=None
       self.ligand_function=None
       self.ligand_class=None

               
       self.charge=None 
       self.Ebound=None   
       self.Eglob=None
       self.Estrain_hartree=None
       self.Estrain_kcal=None  

       self.nconfs=None 
       self.Nrotatable=None
       self.Nheavy=None 
       self.aromatic_prop=None 
       self.Naromatic_rings=None 
       self.Naliphatic_rings=None       
       self.Naliphatic_carbocycles=None 
       self.Naliphatic_heterocycles=None 
       
   

       self.Nacceptor=None 
       self.Ndonor=None 
       self.Nsingleb=None 
       self.Ndoubleb=None  
       self.Ntripleb=None  
       self.Naromaticb=None 

       self.molWt=None 
       self.NPR1=None 
       self.NPR2=None            
       self.global_pair=None

   def load(self,inpath=None,silent=False):

       straindict=pd.read_json(inpath).T
       ligdict=pd.DataFrame.from_dict(straindict)

       self.ligids=np.array(list(ligdict.index))  
       
       self.pdbid=ligdict['pdbid'] 
       self.ligand_code=ligdict['ligand_code'] 
       self.EC_no=ligdict['EC_no'] 
       self.enzyme_class=ligdict['enzyme_class'] 
       self.enzyme_subclass=ligdict['enzyme_subclass'] 
       self.ligand_function=ligdict['ligand_function'] 
       self.ligand_class=ligdict['ligand_class']    
       
       
       self.charge=ligdict['charge']    
       
       self.Ebound=ligdict['Ebound_aimnet']
       self.Eglob=ligdict['Eglob_aimnet']   
       self.Estrain_hartree=ligdict['Estrain_aimnet_hartrees']
       self.Estrain_kcal=ligdict['Estrain_aimnet_kcal']
       
       self.nconfs=ligdict['N_conformers']
       self.Nheavy=ligdict['Nheavy']
       self.aromatic_prop=ligdict['aromatic_prop']
       self.molWt=ligdict['molWt']
       self.NPR1=ligdict['NPR1']
       self.NPR2=ligdict['NPR2']
       self.Naromatic_rings=ligdict['Naromatic_rings']
       self.Naliphatic_rings=ligdict['Naliphatic_rings']
       self.Naliphatic_carbocycles=ligdict['Naliphatic_carbocycles']       
       self.Naliphatic_heterocycles=ligdict['Naliphatic_heterocycles']       
       self.Nacceptor=ligdict['Nacceptor']
       self.Ndonor=ligdict['Ndonor']
       self.Nsingleb=ligdict['Nsingle_bonds']
       self.Ndoubleb=ligdict['Ndouble_bonds'] 
       self.Ntripleb=ligdict['Ntriple_bonds'] 
       self.Naromaticb=ligdict['Naromatic_bonds']
       self.Nrotatable=ligdict['Nrotatable']
       self.global_pair=ligdict['global_pair']  
    


In [ ]:
def select_results(selected_ligs,myresults,insdf):

  E_accpt=[]
  chg_accpt=[]

  arom_accpt=[]
  arom_accpt_group=[]
  aromp_group_vals=np.arange(0.0,1.1,0.2)
  aromp_group_idx=np.arange(0,11,2)
    
  npr1=[]
  npr2=[]

  Nrotlist=[]
  Nrot_group=[]
  gid_label=[]
  group_upper=[[0,2],[3,5],[6,8],[9,11],[12,14],[15,17],[18,20],[21,23]]

  nheavy_accpt=[]
  nheavy_accpt_group=[]   
  nheavy_range=np.arange(0,60,10) 

      
  for l in selected_ligs:
      e=float(myresults.Estrain_kcal[l])
      if(e<0.0): e=0.0
      E_accpt.append(e)
    
      chg_accpt.append(int(myresults.charge[l]))
    
      #aromatic proportion
      ap=myresults.aromatic_prop[l]
      ap_group=[str(r'$arom_{prop}\leq$')+'{0:.1f}'.format(a) for i,a in enumerate(aromp_group_vals) if ap<=a][0]
      for i,a in enumerate(aromp_group_vals):
          if(i==0):
                 if(ap<=aromp_group_vals[i]): 
                      ap_group=str('$AP=$')+'0.0'
          else:         
               if(aromp_group_vals[i-1]<ap<=aromp_group_vals[i]):
                   
                  t0='{0:.1f}'.format(aromp_group_vals[i-1])+str(r'$<$')
                  t1=str('$AP\leq$')
                  t2='{0:.1f}'.format(a)
                  ap_group=t0+t1+t2
                  
      arom_accpt.append(float(ap))
      arom_accpt_group.append(ap_group)
      
      
      #nheavy
      nh=int(myresults.Nheavy[l])
      for i,a in enumerate(nheavy_range):
          if(i==0):
             #print(nh,nheavy_range[i]) 
             if(nh<=nheavy_range[i]): 
                nh_group=str('$N_{heavy}=$')+'0'
          else:         
             if(nheavy_range[i-1]<nh<=nheavy_range[i]):
                  t0='{0:d}'.format(nheavy_range[i-1])+str(r'$<$')
                  t1=str('$N_{heavy}\leq$')
                  t2='{0:d}'.format(a)
                  nh_group=t0+t1+t2
                  
      nheavy_accpt.append(int(nh))
      nheavy_accpt_group.append(nh_group)

      
      #NPR 
      npr1.append(float(myresults.NPR1[l]))
      npr2.append(float(myresults.NPR2[l]))



      #Rotatable bonds
      nrot=myresults.Nrotatable[l]
      Nrotlist.append(nrot)
      gid=[i for i,r in enumerate(group_upper) if r[0]<=nrot<=r[1]][0]

      tle=str(group_upper[gid][0])
      tge=str(group_upper[gid][1]) 
      t1=tle+r'$\leq N_{rot} \leq$'+tge
      gid_label.append(t1)
        
      Nrot_group.append(gid)
  
  outs=[E_accpt,chg_accpt,arom_accpt,arom_accpt_group,npr1,npr2,
        Nrotlist,Nrot_group,gid_label,
        nheavy_accpt,nheavy_accpt_group]  
    
  return outs       


In [ ]:
def plot_ch(E_accpt,chg_accpt,
            ch_inplot=[-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6],        
            xlim_plot=[0,25],ylim=600,
            dpi=150,
            colors=distinct_colors,
            bbox_anchor=(.9, 0.15),
            text_x=7,text_y=550,tinc=50,havetxt=True,tbl_x=0.15,tbl_yoff=-0.55):


    e4plot=[E_accpt[i] for i,ch in enumerate(chg_accpt) if ch in ch_inplot]
    c4plot=[chg_accpt[i] for i,ch in enumerate(chg_accpt) if ch in ch_inplot ] 

    df_chg= pd.DataFrame({'Estrain':e4plot, 'Charge':c4plot})
    df_chg['Charge'] = pd.Categorical(df_chg['Charge'], ch_inplot)

    
    grouped = df_chg.groupby('Charge')

    # Create histogram with discrete bins (bin width is 1), colored by type
    plt.rcParams['figure.dpi'] = dpi
    fig, ax = plt.subplots(figsize=(6,3))
    im=sns.histplot(data=df_chg, x='Estrain', hue='Charge', hue_order=ch_inplot,multiple='dodge', discrete=True,
                    edgecolor='black', stat='count',palette=colors, alpha=0.7,
                    element='bars',kde=True,
                    line_kws={'lw': 1.5, 'ls':'-'})

    ax.set_xticks(np.arange(df_chg['Estrain'].min(), df_chg['Estrain'].max()))
    ax.set_xlim(xlim_plot[0],xlim_plot[1])
    ax.set_ylim(0,ylim)
    # Additional formatting
    sns.despine(offset=5,left=False, right=True, trim=True)
    ax.get_legend().set_frame_on(False)
    fig.autofmt_xdate(rotation=90)
    sns.move_legend(ax, "lower center", bbox_to_anchor=bbox_anchor, ncol=1, title='Charge', frameon=True)

    ax.set_xlabel(r'LCSE (kcal/mol)')

    if(havetxt):

       table_rown= ('Median','Mean','P75','P95')
       table_values=[]
       table_col_colors=[] 
       table_cell_colors=[]  
    
       toff=0.0
       idx=0
       for k,v in charge_stat_dict.items():
           
           if(int(k) in ch_inplot):
               t0='%.1f'%(v['median'])
               t1='%.1f'%(v['mean'])
               t2='%.1f'%(v['p75'])
               t3='%.1f'%(v['p95'])
               table_values.append([t0,t1,t2,t3])

               table_col_colors.append(colors[idx])
               t='median=%2.1f, mean=%2.1f, P95=%2.1f'%(v['median'],v['mean'],v['p95'])
               #plt.text(text_x,text_y-toff, t, fontsize = 6,color=colors[idx],weight='bold')
               toff=toff+tinc
               idx=idx+1

       # Add a table at the bottom of the Axes # 
       for j in range(4): 
           table_cell_colors.append(table_col_colors)           
       table_cell_colors=np.array(table_cell_colors) 
       table_values=np.transpose(np.array(table_values))  
        
       the_table = plt.table(cellText=table_values,
                             rowLabels=table_rown,
                             cellColours=table_cell_colors,
                             rowLoc='left',colLoc='right',edges='closed',
                             loc='bottom',bbox=[tbl_x,tbl_yoff,0.8,0.3],
                             )
       from matplotlib.font_manager import FontProperties
       import six 
       the_table.auto_set_font_size(False)
       the_table.set_fontsize(7) 
       for cell in the_table._cells:
           the_table._cells[cell].set_alpha(.5)
       
       for k, cell in six.iteritems(the_table._cells):
           cell.set_edgecolor('white')
           cell.set_linewidth(0.2)
           cell.set_text_props(fontproperties=FontProperties(family='DejaVu Sans'))

    
    plt.subplots_adjust(left=0.2, bottom=0.05,hspace=1)
    plt.show()



In [ ]:
def plot_aromp(E_accpt,arom_accpt_group,
               xlim_plot=[0,25],ylim=400,
               dpi=150,
               colors=distinct_colors,bbox_anchor=(.9, 0.15),
               text_x=7,text_y=550,tinc=50,havetxt=True,tbl_x=0.15,tbl_yoff=-0.55):

    df_arom = pd.DataFrame({'Estrain':E_accpt, 'Arom_prop_group':arom_accpt_group})

    apord=['0.0$<$$AP\leq$0.2','0.2$<$$AP\leq$0.4','0.4$<$$AP\leq$0.6','0.6$<$$AP\leq$0.8','0.8$<$$AP\leq$1.0']

    df_arom['Arom_prop_group'] = pd.Categorical(df_arom['Arom_prop_group'], apord)

    # Create histogram with discrete bins (bin width is 1), colored by type
    fig, ax = plt.subplots(figsize=(6,3))

    plot_=sns.histplot(data=df_arom, x='Estrain', hue='Arom_prop_group', multiple='dodge', discrete=True,
                   edgecolor='black', stat='count',palette=colors, alpha=0.7,hue_order=apord,
                   element='bars',kde=True,
                   line_kws={'lw': 1.5, 'ls':'-'})

    # Create x ticks covering the range of all integer values of df['value']
    ax.set_xticks(np.arange(df_arom['Estrain'].min(), df_arom['Estrain'].max()))

    ax.set_xlim(xlim_plot[0],xlim_plot[1])
  
    ax.set_ylim(0,ylim)

    ax.set_xlabel(r'LCSE (kcal/mol)') 
    # Additional formatting
    sns.despine(offset=5,left=False, right=True, trim=True)
    ax.get_legend().set_frame_on(False)
    fig.autofmt_xdate(rotation=90)
    sns.move_legend(ax, "lower center", bbox_to_anchor=bbox_anchor, ncol=1, title='Aromatic proportion', frameon=True)

    if(havetxt):
       table_rown= ('Median','Mean','P75','P95')
       table_values=[]
       table_col_colors=[] 
       table_cell_colors=[]  
        
       toff=0.0
       idx=0
       for k,v in arom_stat_dict.items():
           if(k in apord):
               t0='%.1f'%(v['median'])
               t1='%.1f'%(v['mean'])
               t2='%.1f'%(v['p75'])
               t3='%.1f'%(v['p95'])
               table_values.append([t0,t1,t2,t3])
               table_col_colors.append(colors[idx])
               
               #t='q=%d, median=%.1f, mean=%.1f, P95=%.1f'%(int(k),v['median'],v['mean'],v['p95'])
               t='median=%2.1f, mean=%2.1f, P95=%2.1f'%(v['median'],v['mean'],v['p95'])
               #plt.text(text_x,text_y-toff, t, fontsize = 6,color=colors[idx],weight='bold')
               toff=toff+tinc
               idx=idx+1

       # Add a table at the bottom of the Axes # 
       for j in range(4): 
           table_cell_colors.append(table_col_colors)           
       table_cell_colors=np.array(table_cell_colors) 
       table_values=np.transpose(np.array(table_values))  
        
       the_table = plt.table(cellText=table_values,
                             rowLabels=table_rown,
                             cellColours=table_cell_colors,
                             rowLoc='left',colLoc='right',edges='closed',
                             loc='bottom',bbox=[tbl_x,tbl_yoff,0.8,0.3],
                             )
       from matplotlib.font_manager import FontProperties
       import six 
       the_table.auto_set_font_size(False)
       the_table.set_fontsize(7) 
       for cell in the_table._cells:
           the_table._cells[cell].set_alpha(.5)
       
       for k, cell in six.iteritems(the_table._cells):
           cell.set_edgecolor('white')
           cell.set_linewidth(0.2)
           cell.set_text_props(fontproperties=FontProperties(family='DejaVu Sans'))

    plt.subplots_adjust(left=0.2, bottom=0.05,hspace=1)

    plt.show()

In [ ]:
def plot_rotatable(E_accpt,gid_label,
            xlim_plot=[0,25],ylim=650,
            dpi=150,
            colors=distinct_colors,bbox_anchor=(.9, 0.15),
            text_x=7,text_y=550,tinc=50,havetxt=True,tbl_x=0.15,tbl_yoff=-0.55):

    df_nrot = pd.DataFrame({'Estrain': E_accpt, 'Nrotatable': gid_label})
    rotord=['0$\leq N_{rot} \leq$2','3$\leq N_{rot} \leq$5','6$\leq N_{rot} \leq$8','9$\leq N_{rot} \leq$11']
    df_nrot['Nrotatable'] = pd.Categorical(df_nrot['Nrotatable'], rotord)

    
    # Create histogram with discrete bins (bin width is 1), colored by type
    fig, ax = plt.subplots(figsize=(6,3))
    sns.histplot(data=df_nrot, x='Estrain', hue='Nrotatable', multiple='dodge', discrete=True,
             edgecolor='black', stat='count', palette=colors,alpha=0.7,hue_order=rotord,
             element='bars',kde=True,
             line_kws={'lw': 1.5, 'ls':'-'})


    # Create x ticks covering the range of all integer values of df['value']
    ax.set_xticks(np.arange(df_nrot['Estrain'].min(), df_nrot['Estrain'].max()+0.5))
    ax.set_xlim(xlim_plot[0],xlim_plot[1])
    ax.set_ylim(0,ylim)
    ax.set_xlabel(r'LCSE (kcal/mol)')

    # Additional formatting
    sns.despine(offset=5,left=False, right=True, trim=True)
    ax.get_legend().set_frame_on(False)
    fig.autofmt_xdate(rotation=90)
    sns.move_legend(ax, "lower center", bbox_to_anchor=bbox_anchor, ncol=1, title='Rotatable bonds', frameon=True)

    if(havetxt):
       toff=0.0
       idx=0
       table_rown= ('Median','Mean','P75','P95')
       table_values=[]
       table_col_colors=[] 
       table_cell_colors=[]  
        
       for k,v in rotord_stat_dict.items():
           if(k in rotord):
               t0='%.1f'%(v['median'])
               t1='%.1f'%(v['mean'])
               t2='%.1f'%(v['p75'])
               t3='%.1f'%(v['p95'])
               table_values.append([t0,t1,t2,t3])
               table_col_colors.append(colors[idx])
               idx=idx+1
               
       # Add a table at the bottom of the Axes # 
       for j in range(4): 
           table_cell_colors.append(table_col_colors)           
       table_cell_colors=np.array(table_cell_colors) 
       table_values=np.transpose(np.array(table_values))  
        
       the_table = plt.table(cellText=table_values,
                             rowLabels=table_rown,
                             cellColours=table_cell_colors,
                             rowLoc='left',colLoc='right',edges='closed',
                             loc='bottom',bbox=[tbl_x,tbl_yoff,0.8,0.3],
                             )
       from matplotlib.font_manager import FontProperties
       import six 
       the_table.auto_set_font_size(False)
       the_table.set_fontsize(7) 
       for cell in the_table._cells:
           the_table._cells[cell].set_alpha(.5)
       
       for k, cell in six.iteritems(the_table._cells):
           cell.set_edgecolor('white')
           cell.set_linewidth(0.2)
           cell.set_text_props(fontproperties=FontProperties(family='DejaVu Sans'))

    plt.subplots_adjust(left=0.2, bottom=0.05,hspace=1)

    plt.show()

In [ ]:
def plot_heavy(E_accpt,nheavy_accpt_group,
            xlim_plot=[0,25],ylim=650,
            dpi=150,
            colors=distinct_colors,bbox_anchor=(.82, 0.25),horder=None,
            text_x=7,text_y=550,tinc=50,havetxt=True,tbl_x=0.15,tbl_yoff=-0.55):
            
    if(horder is None):        
       heavy_order=['0$<$$N_{heavy}\\leq$10','10$<$$N_{heavy}\\leq$20','20$<$$N_{heavy}\\leq$30','30$<$$N_{heavy}\\leq$40','40$<$$N_{heavy}\\leq$50']

    else:
       heavy_order=horder
    df_nheavy = pd.DataFrame({'Estrain': E_accpt, 'Nheavy': nheavy_accpt_group})
    # Create histogram with discrete bins (bin width is 1), colored by type
    fig, ax = plt.subplots(figsize=(6,3))
    sns.histplot(data=df_nheavy, x='Estrain', hue='Nheavy', multiple='dodge', discrete=True,
             edgecolor='black', stat='count', palette=colors,alpha=0.7,hue_order=heavy_order,
             element='bars',kde=True,
             line_kws={'lw': 1.2, 'ls':'-'})


    # Create x ticks covering the range of all integer values of df['value']
    ax.set_xticks(np.arange(df_nheavy['Estrain'].min(), df_nheavy['Estrain'].max()+0.5))
    ax.set_xlim(xlim_plot[0],xlim_plot[1])
    ax.set_ylim(0,ylim)
    ax.set_xlabel(r'LCSE (kcal/mol)')

    # Additional formatting
    sns.despine(offset=5,left=False, right=True, trim=True)
    ax.get_legend().set_frame_on(False)
    fig.autofmt_xdate(rotation=90)
    sns.move_legend(ax, "lower center", bbox_to_anchor=bbox_anchor, ncol=1, title='Heavy atoms', frameon=True)

    if(havetxt):
       table_rown= ('Median','Mean','P75','P95')
       table_values=[]
       table_col_colors=[] 
       table_cell_colors=[]  
    
        
       toff=0.0
       idx=0
       for k,v in heavy_stat_dict.items():
           if(k in heavy_order):
               t0='%.1f'%(v['median'])
               t1='%.1f'%(v['mean'])
               t2='%.1f'%(v['p75'])
               t3='%.1f'%(v['p95'])
               table_values.append([t0,t1,t2,t3])
               table_col_colors.append(colors[idx])
               idx=idx+1

        
       # Add a table at the bottom of the Axes # 
       for j in range(4): 
           table_cell_colors.append(table_col_colors)           
       table_cell_colors=np.array(table_cell_colors) 
       table_values=np.transpose(np.array(table_values))  
        
       the_table = plt.table(cellText=table_values,
                             rowLabels=table_rown,
                             cellColours=table_cell_colors,
                             rowLoc='left',colLoc='right',edges='closed',
                             loc='bottom',bbox=[tbl_x,tbl_yoff,0.8,0.3],
                             )
       from matplotlib.font_manager import FontProperties
       import six 
       the_table.auto_set_font_size(False)
       the_table.set_fontsize(7) 
       for cell in the_table._cells:
           the_table._cells[cell].set_alpha(.5)
       
       for k, cell in six.iteritems(the_table._cells):
           cell.set_edgecolor('white')
           cell.set_linewidth(0.2)
           cell.set_text_props(fontproperties=FontProperties(family='DejaVu Sans'))


    plt.subplots_adjust(left=0.2, bottom=0.05,hspace=1)

    plt.show()  

In [ ]:
def plot_hist_grid(his2plot,his2plot_keys,
                   n_cols,
                   xlim_plot=[0,32],  
                   indiv=False,
                   colors=distinct_colors,
                   dpi=150,
                   esub=False):


    num_bins=20
    hislen=len(his2plot_keys)
    n_rows=int(np.ceil(hislen/n_cols))

    ecord=['Transferases','Hydrolases','Oxidoreductases','Isomerases','Ligases','Lyases']
    
    ncolsize=n_cols*3  
    nrowsize=n_rows*2
    if(esub):
      ncolsize=n_cols*3  
      nrowsize=n_rows*3  
        
    fig, axes = plt.subplots(nrows=n_rows, 
                             ncols=n_cols, 
                             figsize=(ncolsize,nrowsize))
                             
    fig.subplots_adjust(top=0.8)

    hidx=0
    if(n_rows>1):
      for i in range(n_rows):
        for j in range(n_cols):
            axs=axes[i][j]
            
            label=his2plot_keys[hidx]
            hisdata=his2plot[hidx]
              
            df = pd.DataFrame(hisdata, columns=[label]) 

            hcolor=[colors[hidx]]

            sns.histplot(data=df,bins=25,discrete=True,
                         edgecolor=colors[hidx], stat='count', palette=hcolor,alpha=0.3,
                         kde=True,
                         line_kws={'lw': 2, 'ls':'-','color': 'black'},legend=label,
                         ax=axs)
                         
            axs.lines[0].set_color('black')
            axs.set_xlim(-1,xlim_plot[1])
            axs.set_xlabel(r'LCSE (kcal/mol)')
            hidx=hidx+1
            axes[i][j].set_label(r'LCSE (kcal/mol)')
            if(esub):
               sns.move_legend(axs, "lower center", bbox_to_anchor=(.5, 1.01), ncol=1, frameon=True)



    else:

      for j in range(n_cols):
            axs=axes[j]
            
            label=his2plot_keys[hidx]
            hisdata=his2plot[hidx]
              
            df = pd.DataFrame(hisdata, columns=[label]) 
            hcolor=[colors[hidx]]

            sns.histplot(data=df,bins=25,discrete=True,
                         edgecolor=colors[hidx], stat='count', palette=hcolor,alpha=0.3,
                         kde=True,
                         line_kws={'lw': 2, 'ls':'-','color': 'black'},legend=label,
                         ax=axs)
                         
            axs.lines[0].set_color('black')
            axs.set_xlim(-1,xlim_plot[1])

            hidx=hidx+1
            axes[j].set_label(r'LCSE (kcal/mol)')
            axs.set_xlabel(r'LCSE (kcal/mol)')

            if(esub):
               sns.move_legend(axs, "lower center", bbox_to_anchor=(.5, 1.01), ncol=1, frameon=True)


    
    plt.tight_layout()

    plt.show()


## Load LCSE Dictionary

In [ ]:
from lcse import DATA_DIR
injson = str(DATA_DIR / "LCSE_details.json.gz")   # pandas reads .gz transparently
insdf = str(DATA_DIR / "bound_conformers.sdf.gz")
results = result_loader()
results.load(injson)


## ANALYZE

### ALL LIGANDS

In [ ]:
lig2plot=results.ligids
outs=select_results(lig2plot,results,insdf)

In [ ]:
#to plot only Estrain<=25 kcal/mol
E_accpt=[]
chg_accpt=[]
arom_accpt=[]
arom_accpt_group=[]
npr1=[]
npr2=[]
Nrotlist=[]
Nrot_group=[]
gid_label=[]
nheavy_accpt=[]
nheavy_accpt_group=[]

chg_accp_group=[-3,-2,-1,0,1,2,3]
for i,e in enumerate(outs[0]):
    if(float(e)<=25.0):
       E_accpt.append(outs[0][i])
        
       chg_accpt.append(outs[1][i])  
       arom_accpt.append(outs[2][i])
       arom_accpt_group.append(outs[3][i])
       npr1.append(outs[4][i])
       npr2.append(outs[5][i])
       Nrotlist.append(outs[6][i])
       Nrot_group.append(outs[7][i])
       gid_label.append(outs[8][i]) 
       nheavy_accpt.append(outs[9][i])
       nheavy_accpt_group.append(outs[10][i])


#### Charge

In [ ]:
ch_inplot=[-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6]
charge_stat_dict={}
for aa in ch_inplot:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if chg_accpt[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75)  
       charge_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 


In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-4,-3,-2,-1,0,1,2],
        xlim_plot=[-1,25],ylim=600,
        dpi=600,
        colors=distinct_colors,
        text_x=7,text_y=550,tinc=50,havetxt=True)

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-6,-5,3,4,5],
        xlim_plot=[-1,25],ylim=10,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.75, 0.15),
        havetxt=False)

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-3,-2,-1,0,1,2,3],
        xlim_plot=[-1,25],ylim=600,
        dpi=600,
        colors=distinct_colors,
        text_x=7,text_y=550,tinc=50)

#### Aromatic Proportion

In [ ]:
a=[x for x in arom_accpt_group if x=='0.8$<$$AP\leq$1.0']
apord=['0.0$<$$AP\leq$0.2','0.2$<$$AP\leq$0.4','0.4$<$$AP\leq$0.6','0.6$<$$AP\leq$0.8','0.8$<$$AP\leq$1.0']

arom_stat_dict={}
for aa in apord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if arom_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       arom_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 


In [ ]:
plot_aromp(E_accpt,arom_accpt_group,
        xlim_plot=[-1,25],ylim=500,
        dpi=600,
        colors=distinct_colors, bbox_anchor=(.83, 0.35),
        text_x=6,text_y=480,tinc=50)


#### Rotatable Bonds

In [ ]:
rotord=['0$\leq N_{rot} \leq$2','3$\leq N_{rot} \leq$5','6$\leq N_{rot} \leq$8','9$\leq N_{rot} \leq$11']

rotord_stat_dict={}
for aa in rotord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if gid_label[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       rotord_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

In [ ]:
plot_rotatable(E_accpt,gid_label,
            xlim_plot=[-1,25],ylim=700,
            dpi=600,
            colors=distinct_colors,bbox_anchor=(.85, 0.4),
            text_x=6,text_y=680,tinc=50)

#### Heavy Atoms

In [ ]:
heavy_order=['0$<$$N_{heavy}\\leq$10','10$<$$N_{heavy}\\leq$20','20$<$$N_{heavy}\\leq$30','30$<$$N_{heavy}\\leq$40','40$<$$N_{heavy}\\leq$50']

heavy_stat_dict={}
for aa in heavy_order:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if nheavy_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       heavy_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

In [ ]:
heavy_order=['0$<$$N_{heavy}\\leq$10','10$<$$N_{heavy}\\leq$20','20$<$$N_{heavy}\\leq$30','30$<$$N_{heavy}\\leq$40','40$<$$N_{heavy}\\leq$50']
plot_heavy(E_accpt,nheavy_accpt_group,
            xlim_plot=[-1,25],ylim=500,
            dpi=600,
            colors=distinct_colors,horder=heavy_order,
            text_x=6,text_y=480,tinc=50)


### LIGANDS CLASSIFIED AS NON-COFACTORS

In [ ]:
lig2plot=[l for l in results.ligids if results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None]  

outs=select_results(lig2plot,results,insdf)
E_accpt=outs[0]
chg_accpt=outs[1]
arom_accpt=outs[2]
arom_accpt_group=outs[3]
npr1=outs[4]
npr2=outs[5]
Nrotlist=outs[6]
Nrot_group=outs[7]
gid_label= outs[8]   
nheavy_accpt=outs[9]
nheavy_accpt_group=outs[10]


#### Charge

In [ ]:
ch_inplot=[-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6]
charge_stat_dict={}
for aa in ch_inplot:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if chg_accpt[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p75=np.percentile(a, 75)  
       p95=np.percentile(a, 95) 
       charge_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-3,-2,-1,0,1,2,3],
        xlim_plot=[-1,25],ylim=600,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.92, 0.15),
        text_x=7,text_y=550,tinc=50,havetxt=True)

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-4,-3,-2,-1,0,1,2],
        xlim_plot=[-1,25],ylim=600,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.92, 0.15),
        text_x=7,text_y=550,tinc=50,havetxt=True)

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-6,-5,3,4,5,6],
        xlim_plot=[-1,25],ylim=10,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.75, 0.15),
        text_x=7,text_y=550,tinc=50,havetxt=False)

#### Aromatic Proportion

In [ ]:
apord=['0.0$<$$AP\leq$0.2','0.2$<$$AP\leq$0.4','0.4$<$$AP\leq$0.6','0.6$<$$AP\leq$0.8','0.8$<$$AP\leq$1.0']

arom_stat_dict={}
for aa in apord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if arom_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       arom_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

plot_aromp(E_accpt,arom_accpt_group,
        xlim_plot=[-1,25],ylim=500,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.83, 0.35),
        text_x=7,text_y=480,tinc=50,havetxt=True)

#### Rotatable Bonds

In [ ]:
rotord=['0$\leq N_{rot} \leq$2','3$\leq N_{rot} \leq$5','6$\leq N_{rot} \leq$8','9$\leq N_{rot} \leq$11']

rotord_stat_dict={}
for aa in rotord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if gid_label[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       rotord_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        
plot_rotatable(E_accpt,gid_label,
            xlim_plot=[-1,25],ylim=700,
            dpi=600,
            colors=distinct_colors,bbox_anchor=(.85, 0.25),
            text_x=7,text_y=580,tinc=50,havetxt=True)

#### Heavy Atoms

In [ ]:
heavy_order=['0$<$$N_{heavy}\\leq$10','10$<$$N_{heavy}\\leq$20','20$<$$N_{heavy}\\leq$30','30$<$$N_{heavy}\\leq$40','40$<$$N_{heavy}\\leq$50']

heavy_stat_dict={}
for aa in heavy_order:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if nheavy_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       heavy_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

plot_heavy(E_accpt,nheavy_accpt_group,
            xlim_plot=[-1,25],ylim=500,
            dpi=600,
            colors=distinct_colors,
            text_x=6,text_y=480,tinc=50,havetxt=True)

### LIGANDS CLASSIFIED AS A COFACTOR

In [ ]:
lig2plot=[l for l in results.ligids if results.ligand_function[l]=='cofactor' or results.ligand_function[results.global_pair[l]]=='cofactor']  

outs=select_results(lig2plot,results,insdf)
E_accpt=outs[0]
chg_accpt=outs[1]
arom_accpt=outs[2]
arom_accpt_group=outs[3]
npr1=outs[4]
npr2=outs[5]
Nrotlist=outs[6]
Nrot_group=outs[7]
gid_label= outs[8]   
nheavy_accpt=outs[9]
nheavy_accpt_group=outs[10]


#### Charge

In [ ]:
ch_inplot=[-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6]
charge_stat_dict={}
for aa in ch_inplot:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if chg_accpt[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       charge_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        
#there is no 2,3
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-3,-2,-1,0,1],
        xlim_plot=[-1,25],ylim=60,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.8, 0.15),
        text_x=7,text_y=60,tinc=5,havetxt=True,tbl_x=0.1,tbl_yoff=-0.7)

#### Aromatic Proportion

In [ ]:
apord=['0.0$<$$AP\leq$0.2','0.2$<$$AP\leq$0.4','0.4$<$$AP\leq$0.6','0.6$<$$AP\leq$0.8','0.8$<$$AP\leq$1.0']

arom_stat_dict={}
for aa in apord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if arom_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95)
       p75=np.percentile(a, 75) 
       arom_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 

plot_aromp(E_accpt,arom_accpt_group,
        xlim_plot=[-1,25],ylim=60,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.72, 0.35),
        text_x=3,text_y=60,tinc=5,havetxt=True,tbl_x=0.1,tbl_yoff=-0.7)

#### Rotatable Bonds


In [ ]:
rotord=['0$\leq N_{rot} \leq$2','3$\leq N_{rot} \leq$5','6$\leq N_{rot} \leq$8','9$\leq N_{rot} \leq$11']

rotord_stat_dict={}
for aa in rotord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if gid_label[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       rotord_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 

plot_rotatable(E_accpt,gid_label,
            xlim_plot=[-1,25],ylim=60,
            dpi=600,
            colors=distinct_colors,bbox_anchor=(.72, 0.35),
            text_x=3,text_y=60,tinc=5,havetxt=True,tbl_x=0.1,tbl_yoff=-0.7)

#### Heavy Atoms

In [ ]:
heavy_order=['0$<$$N_{heavy}\\leq$10','10$<$$N_{heavy}\\leq$20','20$<$$N_{heavy}\\leq$30','30$<$$N_{heavy}\\leq$40','40$<$$N_{heavy}\\leq$50']

heavy_stat_dict={}
for aa in heavy_order:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if nheavy_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75) 
       heavy_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

plot_heavy(E_accpt,nheavy_accpt_group,
            xlim_plot=[-1,25],ylim=60,
            dpi=600,
            colors=distinct_colors,bbox_anchor=(.72, 0.35),
            text_x=3,text_y=60,tinc=5,havetxt=True,tbl_x=0.1,tbl_yoff=-0.7)

### LIGANDS WITH SINGLE TARGET

In [ ]:
lig_not_uniqs=[l for l in results.ligids if l!=results.global_pair[l]]
gp=[results.global_pair[l] for l in results.ligids if l!=results.global_pair[l]]
global_pairs=list(set(gp))
duplicate_ligands=[x for x in lig_not_uniqs]   
duplicate_ligands.extend(global_pairs)

lig_uniqs=[l for l in results.ligids if l not in duplicate_ligands]

print('Nduplicates: %d'%(len(duplicate_ligands)))
print('Nuniqs: %d'%(len(lig_uniqs)))

In [ ]:
outs=select_results(lig_uniqs,results,insdf)
E_accpt=outs[0]
chg_accpt=outs[1]
arom_accpt=outs[2]
arom_accpt_group=outs[3]
npr1=outs[4]
npr2=outs[5]
Nrotlist=outs[6]
Nrot_group=outs[7]
gid_label= outs[8]   
nheavy_accpt=outs[9]
nheavy_accpt_group=outs[10]


#### Charge

In [ ]:
ch_inplot=[-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6]
charge_stat_dict={}
for aa in ch_inplot:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if chg_accpt[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75)  
       charge_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-3,-2,-1,0,1,2,3],
        xlim_plot=[-1,25],ylim=500,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.92, 0.15),
        text_x=7,text_y=550,tinc=50,havetxt=True,tbl_yoff=-0.6)

In [ ]:
plot_ch(E_accpt,chg_accpt,
        ch_inplot=[-4,-3,-2,-1,0,1,2],
        xlim_plot=[-1,25],ylim=500,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.92, 0.15),
        text_x=7,text_y=550,tinc=50,havetxt=True,tbl_yoff=-0.6)


In [ ]:
ch_inplot=[-6,-5,3,4,5,6]
my_ch_inplot=[]
for aa in ch_inplot:
    if aa in chg_accpt and aa not in my_ch_inplot: my_ch_inplot.append(aa)
if(len(my_ch_inplot)!=0):        
   plot_ch(E_accpt,chg_accpt,
           ch_inplot=my_ch_inplot,
           xlim_plot=[-1,25],ylim=10,
           dpi=600,
           colors=distinct_colors,bbox_anchor=(.7, 0.2),
           text_x=7,text_y=550,tinc=50,havetxt=False,tbl_yoff=-0.6)

#### Aromatic Proportion

In [ ]:
apord=['0.0$<$$AP\leq$0.2','0.2$<$$AP\leq$0.4','0.4$<$$AP\leq$0.6','0.6$<$$AP\leq$0.8','0.8$<$$AP\leq$1.0']

arom_stat_dict={}
for aa in apord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if arom_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75)  
       arom_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        
plot_aromp(E_accpt,arom_accpt_group,
        xlim_plot=[-1,25],ylim=350,
        dpi=600,
        colors=distinct_colors,bbox_anchor=(.83, 0.35),
        text_x=7,text_y=380,tinc=50,havetxt=True,tbl_yoff=-0.6)

#### Rotatable Bonds

In [ ]:
rotord=['0$\leq N_{rot} \leq$2','3$\leq N_{rot} \leq$5','6$\leq N_{rot} \leq$8','9$\leq N_{rot} \leq$11']

rotord_stat_dict={}
for aa in rotord:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if gid_label[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75)  
       rotord_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 

plot_rotatable(E_accpt,gid_label,
            xlim_plot=[-1,25],ylim=500,
            dpi=600,
            colors=distinct_colors,bbox_anchor=(.85, 0.45),
            text_x=7,text_y=490,tinc=50,havetxt=True,tbl_yoff=-0.6)

#### Heavy Atoms

In [ ]:
heavy_order=['0$<$$N_{heavy}\\leq$10','10$<$$N_{heavy}\\leq$20','20$<$$N_{heavy}\\leq$30','30$<$$N_{heavy}\\leq$40','40$<$$N_{heavy}\\leq$50']

heavy_stat_dict={}
for aa in heavy_order:
    a=np.array([float(x) for i,x in enumerate(E_accpt) if nheavy_accpt_group[i]==aa])
    if(len(a)>0):
       med=statistics.median(a)   
       p95=np.percentile(a, 95) 
       p75=np.percentile(a, 75)  
       heavy_stat_dict[str(aa)]={'min':min(a),'max':max(a),'median': med,'mean': np.mean(a),'p95':p95,'p75':p75} 
        
plot_heavy(E_accpt,nheavy_accpt_group,
            xlim_plot=[-1,25],ylim=400,
            dpi=600,
            colors=distinct_colors,bbox_anchor=(.83, 0.3),
            text_x=7,text_y=490,tinc=50,havetxt=True,tbl_yoff=-0.6)

### LIGANDS WITH MULTIPLE TARGETS

In [ ]:
#change colors
distinct_colors=[(0.0, 1.0, 0.0), (1.0, 0.0, 1.0), (0.0, 0.5, 1.0), (1.0, 0.5, 0.0), (0.5, 0.75, 0.5), (0.3903118052289736, 0.13578323628122246, 0.5999178496672565), (0.8320791939881281, 0.007401743231903457, 0.045107915070166316), (0.00896035734809908, 0.9900272566966499, 0.8263061971694814), (0.011787227663116129, 0.47081677328966576, 0.23125008058698604), (0.9737237511698383, 0.5017334102913582, 0.8623283865202347), (1.0, 1.0, 0.0), (0.0, 0.0, 1.0), (0.5, 0.5, 0.0), (0.5307174489739835, 0.657762148707971, 0.9934338306526526), (0.7906368060529793, 0.3251877368042878, 0.43435693306922474), (0.9755227754462751, 0.7934429767069803, 0.47850305531046644), (0.541132699213533, 0.9628240668584229, 0.0960763541927675), (0.026057470782568037, 0.8358530555625399, 0.38603889272002845), (0.6359845567900916, 0.23493171239350275, 0.9772772998391184), (0.40119419798364464, 0.172272635791354, 0.15474157748674977), (1.0, 0.0, 0.5), (0.22200437890312197, 0.48505100086211406, 0.5985190211391912), (0.5, 1.0, 1.0), (0.0, 0.0, 0.5), (0.7244846518272101, 0.7032408807054246, 0.1836836448437018), (0.022701323583258604, 0.23968816743651733, 0.7713478718931338), (0.72691812534969, 0.9677997847594566, 0.633782095965687), (0.1717404098614509, 0.7145661565708478, 0.020246027261439203), (0.21715130609969713, 0.7206339659295353, 0.8105418241298402), (0.7267319808844698, 0.5914102425460385, 0.6668207936347914)]


In [ ]:
lig_not_uniqs=[l for l in results.ligids if l!=results.global_pair[l]]
gp=[results.global_pair[l] for l in results.ligids if l!=results.global_pair[l]]
global_pairs=list(set(gp))
duplicate_ligands=[x for x in lig_not_uniqs]   
duplicate_ligands.extend(global_pairs)

lig_uniqs=[l for l in results.ligids if l not in duplicate_ligands]

print('Nduplicates: %d'%(len(duplicate_ligands)))
print('Nuniqs: %d'%(len(lig_uniqs)))

In [ ]:
max_num=20
ener_dupl_dict={}
lig_dupl_dict={}
for l in duplicate_ligands:
    dupl_list=[x for x in results.ligids if results.global_pair[x]==l]
    if(len(dupl_list)>max_num):   
        ener=[results.Estrain_kcal[x] for x in dupl_list]
        lig_dupl_dict[l]=dupl_list
        ener_dupl_dict[l]=ener        


In [ ]:
his2plot=[]
his2plot_keys=[]

for i,key in enumerate(ener_dupl_dict.keys()): 
        his2plot.append(ener_dupl_dict[key])
        his2plot_keys.append(key.split('_')[0])


plot_hist_grid(his2plot,his2plot_keys,4,indiv=False,
               colors=distinct_colors,
               dpi=600)


### ENZYME CLASSES FOR NON-COFACTOR CLASS LIGANDS

In [ ]:
eclasses=[results.enzyme_class[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l] is not None]
print('Non-cofactors with enzyme class: %d'%(len(eclasses)))
ectypes=ecord=['Transferases','Oxidoreductases','Isomerases','Hydrolases','Ligases','Lyases'] #list(set(eclasses))
ec_strains={}
for e in ectypes:
    strain=[results.Estrain_kcal[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e]
    estrain=[]
    for x in strain:
        if(x<0): ener=0.0
        else: ener=x
        estrain.append(ener)
        
    ec_strains[e]=estrain


#### Main EC

In [ ]:
his2plot=[]
his2plot_keys=[]

for i,key in enumerate(ec_strains.keys()):
    if len(ec_strains[key])>=30:   
        his2plot.append(ec_strains[key])
        his2plot_keys.append(key)


plot_hist_grid(his2plot,his2plot_keys,3,indiv=False,
               colors=distinct_colors,
               dpi=600)



#### Enzyme Subclasses

In [ ]:
eclasses=[results.enzyme_class[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l] is not None]
ectypes=list(set(eclasses))

for e in ectypes:
    ec_no=[results.EC_no[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e]
    ec_sub=list(set(ec_no))
    ec_strains={}
    
    
    for e2 in ec_sub:
        estrain=[]
        strain=[results.Estrain_kcal[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e and results.EC_no[l]==e2]    
        for x in strain:
            if(x<0): ener=0.0
            else: ener=x
            estrain.append(ener)
        if(len(estrain)>30):
           ec_strains[e+'_'+str(e2)]=estrain
    
    his2plot=[]
    his2plot_keys=[]

    for i,key in enumerate(ec_strains.keys()):
        #print(key,len(ec_strains[key]))   
        his2plot.append(ec_strains[key])
        his2plot_keys.append(key)

    if(len(his2plot_keys)>5): ncol=3
    else: ncol=len(his2plot_keys) 


    if(ncol!=0):
       print("%10s   %10s"%("Subclass","Nentry")) 
       for ii,xxx in  enumerate(his2plot_keys):
           #print(xxx, len(his2plot[ii]))
           print("%10s   %10d"%(xxx, len(his2plot[ii])))
           
       plot_hist_grid(his2plot,his2plot_keys,ncol,indiv=False,
                    colors=distinct_colors,
                    dpi=600)
       print("\n")
       print("%s\n"%("="*50))

#### Kinases

In [ ]:
ectypes=['Transferases']#list(set(eclasses))
eclasses=[results.enzyme_class[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]in ectypes]


for e in ectypes:
    ec_no=[results.EC_no[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e]
    ec_sub=list(set(ec_no))
    ec_sub=['2.7'] #,'3.4']
    ec_strains={}
    
    
    for e2 in ec_sub:
        estrain=[]
        strain=[results.Estrain_kcal[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e and results.EC_no[l]==e2]    
        for x in strain:
            if(x<0): ener=0.0
            else: ener=x
            estrain.append(ener)
        if(len(estrain)>30):
           ec_strains[e+'_'+str(e2)]=estrain
    
    his2plot=[]
    his2plot_keys=[]

    for i,key in enumerate(ec_strains.keys()):
        print(key,len(ec_strains[key]))   
        his2plot.append(ec_strains[key])
        his2plot_keys.append(key)


    num_bins=20
    ecord=ectypes
    fig, ax = plt.subplots(figsize=(3,3))
    hidx=0
    label='Kinases' #his2plot_keys[hidx]
    hisdata=his2plot[hidx]
    df = pd.DataFrame(hisdata, columns=[label])
    hcolor=["#F06292"]
    sns.histplot(data=df,bins=25,discrete=True,
                 edgecolor="#F06292", stat='count', palette=hcolor,alpha=0.3,
                 kde=True,
                 line_kws={'lw': 2, 'ls':'-','color': 'black'},legend=label)

    med=statistics.median(hisdata)
    p75=np.percentile(np.array(hisdata), 75) 
    p95=np.percentile(np.array(hisdata), 95) 
    mean=np.mean(np.array(hisdata))
    t0='%-6s = %3.1f'%('median',med)
    t1='%-7s = %3.1f'%('mean',mean)
    t2='%-9s = %3.1f'%('P75',p75)
    t3='%-9s = %3.1f'%('P95',p95)
    ax.text(10,200, t0, fontsize = 10,color='black')
    ax.text(10,180, t1, fontsize = 10,color='black')
    ax.text(10,160, t2, fontsize = 10,color='black')
    ax.text(10,140, t3, fontsize = 10,color='black') 
    
    ax.lines[0].set_color('black')
    ax.set_xlim(-1,31)
    ax.set_label('Estrain')
    ax.set_xlabel(r'LCSE (kcal/mol)')    
    sns.move_legend(ax, "lower center", bbox_to_anchor=(.73, 0.83), ncol=1, frameon=True)
    plt.tight_layout()
 
    plt.show()



#### Glycosyltransferases

In [ ]:
ectypes=['Transferases']#list(set(eclasses))
eclasses=[results.enzyme_class[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]in ectypes]


for e in ectypes:
    ec_no=[results.EC_no[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e]
    ec_sub=list(set(ec_no))
    ec_sub=['2.4'] #,'3.4']
    ec_strains={}
    
    
    for e2 in ec_sub:
        estrain=[]
        strain=[results.Estrain_kcal[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e and results.EC_no[l]==e2]    
        for x in strain:
            if(x<0): ener=0.0
            else: ener=x
            estrain.append(ener)
        if(len(estrain)>30):
           ec_strains[e+'_'+str(e2)]=estrain
    
    his2plot=[]
    his2plot_keys=[]

    for i,key in enumerate(ec_strains.keys()):
        print(key,len(ec_strains[key]))   
        his2plot.append(ec_strains[key])
        his2plot_keys.append(key)


    num_bins=20
    ecord=ectypes
    fig, ax = plt.subplots(figsize=(3,3))
    hidx=0
    label='Glycosyltransferases' #his2plot_keys[hidx]
    hisdata=his2plot[hidx]
    df = pd.DataFrame(hisdata, columns=[label])
    hcolor=["#F06292"]
    sns.histplot(data=df,bins=25,discrete=True,
                 edgecolor="#F06292", stat='count', palette=hcolor,alpha=0.3,
                 kde=True,
                 line_kws={'lw': 2, 'ls':'-','color': 'black'},legend=label)


    med=statistics.median(hisdata)
    p75=np.percentile(np.array(hisdata), 75) 
    p95=np.percentile(np.array(hisdata), 95) 
    mean=np.mean(np.array(hisdata))
    t0='%-6s = %3.1f'%('median',med)
    t1='%-7s = %3.1f'%('mean',mean)
    t2='%-9s = %3.1f'%('P75',p75)
    t3='%-9s = %3.1f'%('P95',p95)
    ax.text(10,55, t0, fontsize = 10,color='black')
    ax.text(10,50, t1, fontsize = 10,color='black')
    ax.text(10,45, t2, fontsize = 10,color='black')
    ax.text(10,40, t3, fontsize = 10,color='black') 
    
    ax.lines[0].set_color('black')
    ax.set_xlim(-1,31)
    ax.set_label('Estrain')
    ax.set_xlabel(r'LCSE (kcal/mol)')    
    sns.move_legend(ax, "lower center", bbox_to_anchor=(.53, 0.83), ncol=1, frameon=True)
    plt.tight_layout()

    plt.show()



#### Peptidases

In [ ]:
ectypes=['Hydrolases']#list(set(eclasses))
eclasses=[results.enzyme_class[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]in ectypes]


for e in ectypes:
    ec_no=[results.EC_no[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e]
    ec_sub=list(set(ec_no))
    ec_sub=['3.4'] #,'3.4']
    ec_strains={}
    
    
    for e2 in ec_sub:
        estrain=[]
        strain=[results.Estrain_kcal[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e and results.EC_no[l]==e2]    
        for x in strain:
            if(x<0): ener=0.0
            else: ener=x
            estrain.append(ener)
        if(len(estrain)>30):
           ec_strains[e+'_'+str(e2)]=estrain
    
    his2plot=[]
    his2plot_keys=[]

    for i,key in enumerate(ec_strains.keys()):
        print(key,len(ec_strains[key]))   
        his2plot.append(ec_strains[key])
        his2plot_keys.append(key)


    num_bins=20
    ecord=ectypes
    fig, ax = plt.subplots(figsize=(3,3))
    hidx=0
    label='Peptidases' #his2plot_keys[hidx]
    hisdata=his2plot[hidx]
    df = pd.DataFrame(hisdata, columns=[label])
    hcolor=["#7E57C2"]
    sns.histplot(data=df,bins=25,discrete=True,
                 edgecolor="#7E57C2", stat='count', palette=hcolor,alpha=0.3,
                 kde=True,
                 line_kws={'lw': 2, 'ls':'-','color': 'black'},legend=label)


    med=statistics.median(hisdata)
    p75=np.percentile(np.array(hisdata), 75) 
    p95=np.percentile(np.array(hisdata), 95) 
    mean=np.mean(np.array(hisdata))
    t0='%-6s = %3.1f'%('median',med)
    t1='%-7s = %3.1f'%('mean',mean)
    t2='%-9s = %3.1f'%('P75',p75)
    t3='%-9s = %3.1f'%('P95',p95)
    ax.text(10,50, t0, fontsize = 10,color='black')
    ax.text(10,45, t1, fontsize = 10,color='black')
    ax.text(10,40, t2, fontsize = 10,color='black')
    ax.text(10,35, t3, fontsize = 10,color='black') 
    
    ax.lines[0].set_color('black')
    ax.set_xlim(-1,30)
    ax.set_label('Estrain')
    ax.set_xlabel(r'LCSE (kcal/mol)')    
    sns.move_legend(ax, "lower center", bbox_to_anchor=(.68, 0.83), ncol=1, frameon=True)
    plt.tight_layout()

    plt.show()



#### Glycosylases

In [ ]:
ectypes=['Hydrolases']#list(set(eclasses))
eclasses=[results.enzyme_class[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]in ectypes]


for e in ectypes:
    ec_no=[results.EC_no[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e]
    ec_sub=list(set(ec_no))
    ec_sub=['3.2'] #,'3.4']
    ec_strains={}
    
    
    for e2 in ec_sub:
        estrain=[]
        strain=[results.Estrain_kcal[l] for l in results.ligids if (results.ligand_function[l] is None or results.ligand_function[results.global_pair[l]] is None) and results.enzyme_class[l]==e and results.EC_no[l]==e2]    
        for x in strain:
            if(x<0): ener=0.0
            else: ener=x
            estrain.append(ener)
        if(len(estrain)>30):
           ec_strains[e+'_'+str(e2)]=estrain
    
    his2plot=[]
    his2plot_keys=[]

    for i,key in enumerate(ec_strains.keys()):
        print(key,len(ec_strains[key]))   
        his2plot.append(ec_strains[key])
        his2plot_keys.append(key)


    num_bins=20
    ecord=ectypes
    fig, ax = plt.subplots(figsize=(3,3))
    hidx=0
    label='Glycosylases' #his2plot_keys[hidx]
    hisdata=his2plot[hidx]
    df = pd.DataFrame(hisdata, columns=[label])
    hcolor=["#7E57C2"]
    sns.histplot(data=df,bins=25,discrete=True,
                 edgecolor="#7E57C2", stat='count', palette=hcolor,alpha=0.3,
                 kde=True,
                 line_kws={'lw': 2, 'ls':'-','color': 'black'},legend=label)


    med=statistics.median(hisdata)
    p75=np.percentile(np.array(hisdata), 75) 
    p95=np.percentile(np.array(hisdata), 95) 
    mean=np.mean(np.array(hisdata))
    t0='%-6s = %3.1f'%('median',med)
    t1='%-7s = %3.1f'%('mean',mean)
    t2='%-9s = %3.1f'%('P75',p75)
    t3='%-9s = %3.1f'%('P95',p95)
    ax.text(10,45, t0, fontsize = 10,color='black')
    ax.text(10,40, t1, fontsize = 10,color='black')
    ax.text(10,35, t2, fontsize = 10,color='black')
    ax.text(10,30, t3, fontsize = 10,color='black') 
    
    ax.lines[0].set_color('black')
    ax.set_xlim(-1,25)
    ax.set_label('Estrain')
    ax.set_xlabel(r'LCSE (kcal/mol)')    
    sns.move_legend(ax, "lower center", bbox_to_anchor=(.65, 0.83), ncol=1, frameon=True)
    plt.tight_layout()
    plt.show()

